# 01 · Error inheritance — does the next model repeat its predecessor's verified mistakes?

**Primary detector of Layer 1** (paper §3.3.1). The loop's defining act is that a wrong high score
becomes a training positive; its most direct observable signature is **inheritance**.

**Design (v1 → v2).** v1's segmented rule creates the overlap band `(0.75, 0.85]`: **mobile**
vehicles scoring there were **garaged and independently assessed**, while immobile vehicles at the
same scores were scrapped and minted forced positives. Among mobile, garage-verified band rows,
those the engineer judged **repairable** (`observed = 0`) are rows where a high v1 score is *known*
to have been wrong. If v2 was trained on v1's forced labels, v2 should **over-score exactly these
rows** — an excess lift in `E[s_v2 | s_v1]` switching on at `0.75`, on the `Y=0` population, beyond
the `Y=1` baseline (both models track genuine severity even without a loop, so the shared-signal
slope is netted out by the second difference).

**No common feature space is needed**: the test consumes only each model's *score*, joined per
claim. The two models' feature matrices never meet.

**Splits are POOLED here, deliberately.** The per-split artefacts are loaded with
`split=config.ALL_SPLITS`, which reads every split and adds a `split` column. This test is a
*population* statement — which claims sat in v1's overlap band and were garage-verified — and a
claim's train/test membership is an artefact of each version's own fitting, unrelated to whether
its label was forced. Restricting to one split would only shrink the band, which the n-gate below
already struggles with. The `split` column is dropped before any cross-version join: v1's split
names and v2's mean different things, so joining on them would silently drop real rows.

**Inputs (all resolved via `src/config.py` — a missing path fails loudly with the entry to fix):**
1. v1 log rows with score, mobility, decision, verified outcome (`loaders.load('v1', split=…)`).
2. v2 scores for the same claims (`loaders.load('v2', split=…).scores` — v2's window sits inside
   v1's production era, so `scoring/predict.py` can score them).
3. Band row counts must pass the n-gate below before any estimate is read.

**Outputs:** paper `[[FIG 4.0]]` and `[[TAB 4.0]]`.

*Kernel: the analysis `.venv` (`python3`). Parquet only — no model pickle is opened here.*

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import figstyle
from loaders import version_data as loaders

figstyle.apply()

FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ---- analysis parameters (the only knobs in this notebook) -----------------------------
MIN_N_SIDE = 30     # minimum rows on EACH side of a cutoff for a jump estimate
MIN_N_BAND = 200    # n-gate: minimum mobile garage-verified rows in the band overall
BANDWIDTH = 0.10    # local window each side of the cutoff
N_BOOT = 2000       # bootstrap draws for the CI
SEED = 20260802
PLACEBOS = (0.55, 0.65)   # cutoffs where no forced minting started -> beta ~ 0 required

## 1 · Load and join — v1's verified world, v2's scores

Everything comes through the loader; nothing here names a path, a raw column or a cutoff.

In [ ]:
# ALL_SPLITS: every split pooled, with a `split` column — see the note above on why this test
# is a population statement rather than a per-split one.
POOLED = config.ALL_SPLITS

v1 = loaders.load("v1", split=POOLED)
v2 = loaders.load("v2", split=POOLED)

rules = config.DECISION_RULES["v1"]
LO, HI = rules["overlap_band"]          # (0.75, 0.85) as declared in config, never hard-coded
MOBILE_VALUES = set(rules["mobile_values"])

d1 = v1.frame  # targets + v1 scores, canonical names, joined on claim_id
if schema.MOBILITY not in d1.columns:
    d1 = d1.merge(
        v1.log[[schema.CLAIM_ID, schema.MOBILITY]].drop_duplicates(schema.CLAIM_ID),
        on=schema.CLAIM_ID, how="left",
    )

# drop v2's split marker before joining: "test" means different things to v1 and v2, so keeping
# both would either collide on the column name or, if joined on, drop rows that genuinely match.
df = d1.merge(v2.scores.drop(columns="split", errors="ignore"),
              on=schema.CLAIM_ID, how="inner")
S1, S2 = v1.score_col, v2.score_col

print(f"v1 rows: {len(d1):,}")
print(f"joined with v2 scores: {len(df):,}  "
      f"({len(df)/max(len(d1),1):.1%} coverage — if low, run scoring/predict.py for v2 on v1-era claims)")

df["mobile"] = df[schema.MOBILITY].isin(MOBILE_VALUES)
df["verified"] = df[schema.DECISION].eq(0)   # garage-routed -> `observed` is the engineer assessment
print(f"mobility present: {df[schema.MOBILITY].notna().mean():.1%} of joined rows")

## 2 · Region table — sanity check against the documented rule

Expected (paper §2.3.1): below 0.75 both segments garage; in `(0.75, 0.85]` immobile scrap /
mobile garage; above 0.85 both scrap. Deviations are reported, not silently dropped — a large
deviation means the logged decisions do not follow the documented rule and the design premise
must be revisited before anything else in this notebook is read.

In [ ]:
df["region"] = pd.cut(
    df[S1], bins=[-np.inf, LO, HI, np.inf],
    labels=[f"below {LO}", f"band ({LO}, {HI}]", f"above {HI}"],
)

table = (
    df.groupby(["region", "mobile", schema.DECISION], observed=True)
      .size().unstack(fill_value=0)
)
print(table)

band = df[(df[S1] > LO) & (df[S1] <= HI)]
viol_mobile = band[band["mobile"] & band[schema.DECISION].eq(1)]
viol_immob = band[~band["mobile"] & band[schema.DECISION].eq(0)]
print(f"\nband rows: {len(band):,}")
print(f"rule deviations — mobile scrapped in band: {len(viol_mobile):,}; "
      f"immobile garaged in band: {len(viol_immob):,}")

## 3 · n-gate

Counts first, analysis second (paper §3.2). If the gate fails, the notebook stops here and the
failure itself is the reported result.

In [ ]:
mv = df[df["mobile"] & df["verified"]].copy()   # mobile, garage-verified: genuine Y observed
mv["Y"] = mv[schema.OBSERVED].astype(int)

in_band = mv[(mv[S1] > LO) & (mv[S1] <= HI)]
counts = {
    "mobile verified, band, Y=0 (verified repairable)": int((in_band["Y"] == 0).sum()),
    "mobile verified, band, Y=1 (verified total loss)": int((in_band["Y"] == 1).sum()),
    "mobile verified, control window just below": int(
        ((mv[S1] > LO - BANDWIDTH) & (mv[S1] <= LO)).sum()
    ),
}
for k, v in counts.items():
    print(f"{k:55s} {v:>8,}")

GATE_PASSED = len(in_band) >= MIN_N_BAND
print(f"\nn-gate ({MIN_N_BAND} mobile verified band rows): {'PASS' if GATE_PASSED else 'FAIL'}")
if not GATE_PASSED:
    print("STOP: the band is too thin to estimate inheritance. Report the counts above as the\n"
          "result (paper §4.1, TAB 4.0 row 'v1→v2' = 'underpowered, n=...').")

## 4 · The jump estimator

Local-linear fit of `s_v2` on `s_v1` on each side of a cutoff within `BANDWIDTH`;
`beta = right(cutoff) − left(cutoff)`. Bootstrap percentile CI over rows.
The inheritance signature is `beta > 0` on `Y=0`, in excess of the `Y=1` baseline, with
`beta ≈ 0` at the placebo cutoffs.

In [ ]:
rng = np.random.default_rng(SEED)


def jump_at(data: pd.DataFrame, cutoff: float, h: float = BANDWIDTH):
    """Local-linear jump of E[s_v2 | s_v1] at `cutoff`. Returns (beta, n_left, n_right) or None."""
    left = data[(data[S1] >= cutoff - h) & (data[S1] < cutoff)]
    right = data[(data[S1] >= cutoff) & (data[S1] <= cutoff + h)]
    if len(left) < MIN_N_SIDE or len(right) < MIN_N_SIDE:
        return None
    bl = np.polyfit(left[S1], left[S2], 1)
    br = np.polyfit(right[S1], right[S2], 1)
    beta = np.polyval(br, cutoff) - np.polyval(bl, cutoff)
    return float(beta), len(left), len(right)


def jump_ci(data: pd.DataFrame, cutoff: float, n_boot: int = N_BOOT):
    point = jump_at(data, cutoff)
    if point is None:
        return None
    beta, n_l, n_r = point
    boots = []
    for _ in range(n_boot):
        sample = data.sample(frac=1.0, replace=True, random_state=rng.integers(2**31))
        b = jump_at(sample, cutoff)
        if b is not None:
            boots.append(b[0])
    lo_ci, hi_ci = (np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan))
    return {"beta": beta, "ci_lo": float(lo_ci), "ci_hi": float(hi_ci),
            "n_left": n_l, "n_right": n_r, "n_boot_ok": len(boots)}


rows = []
if GATE_PASSED:
    for y in (0, 1):
        pop = mv[mv["Y"] == y]
        est = jump_ci(pop, LO)
        rows.append({"pair": "v1→v2", "population": f"Y={y}", "cutoff": LO, **(est or {})})
        for pc in PLACEBOS:
            est_p = jump_ci(pop, pc, n_boot=N_BOOT // 4)
            rows.append({"pair": "v1→v2", "population": f"Y={y} placebo", "cutoff": pc,
                         **(est_p or {})})

results = pd.DataFrame(rows)
if len(results):
    b0 = results.query("population == 'Y=0' and cutoff == @LO")["beta"]
    b1 = results.query("population == 'Y=1' and cutoff == @LO")["beta"]
    if len(b0) and len(b1) and b0.notna().all() and b1.notna().all():
        results = pd.concat([results, pd.DataFrame([{
            "pair": "v1→v2", "population": "second difference (Y0 − Y1)", "cutoff": LO,
            "beta": float(b0.iloc[0] - b1.iloc[0]),
        }])], ignore_index=True)
results

**Read the table as:** inheritance is supported only if the `Y=0` beta at the real cutoff is
positive with a CI excluding 0, **and** exceeds the `Y=1` baseline, **and** the placebo betas are
compatible with 0. Any other pattern is reported as a null or as underpowered — a null here is a
real, publishable result (paper §4.1).

## 5 · Figure — E[s_v2 | s_v1] around the band ([[FIG 4.0]])

In [ ]:
if GATE_PASSED:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    edges = np.arange(0.30, min(HI + 0.05, 1.0) + 1e-9, 0.025)
    for ax, y in zip(axes, (0, 1)):
        pop = mv[mv["Y"] == y]
        binned = pop.groupby(pd.cut(pop[S1], edges), observed=True).agg(
            x=(S1, "mean"), m=(S2, "mean"), n=(S2, "size"),
        ).dropna()
        ax.scatter(binned["x"], binned["m"], s=np.clip(binned["n"], 8, 80))
        for c, style in ((LO, "-"), (HI, "--")):
            ax.axvline(c, linestyle=style, linewidth=1)
        ax.set_title(f"mobile garage-verified, Y={y}"
                     + ("  (verified repairable — the treated test)" if y == 0 else "  (baseline)"))
        ax.set_xlabel("$s_{v1}$ (logged v1 score)")
    axes[0].set_ylabel("mean $s_{v2}$")
    fig.suptitle("Error inheritance: does v2 over-score v1's verified-repairable band rows?")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "v1v2_01_error_inheritance.png", dpi=200, bbox_inches="tight")
    plt.show()

## 6 · v2 → v3 analogue (weaker — reported with its caveat)

v2 has no covariate-segmented band, and its temporal band `(0.825, 0.872]` post-dates v3's
training window, so **no verified rows exist above the cutoff** for this pair. The analogue runs
the excess-tracking comparison on the near-boundary garage region *below* v2's pre-break cutoff:
if v3 inherited v2's forced positives, the gap `E[s_v3 | Y=1] − E[s_v3 | Y=0]` should **narrow as
`s_v2` approaches the cutoff** faster than its mid-score trend (v3 starts treating verified
repairables like total losses where v2 was most confident). No above-cutoff contrast exists —
state that limitation wherever this section is quoted.

In [ ]:
try:
    v3 = loaders.load("v3", split=POOLED)
    d2 = v2.frame.merge(v3.scores.drop(columns="split", errors="ignore"),
                        on=schema.CLAIM_ID, how="inner")
    S3 = v3.score_col

    # ONE regime only. v2's cutoff moved FOUR times (§2.3.2), so pooled rows carry up to five
    # different treatment assignments — and two of the regimes share 0.872 without being the same
    # deployment, so a threshold value cannot identify a regime. Label each row by its regime's
    # (from, until) span and keep the single most populated one; positional indexing into
    # `regimes` would silently pick the open-ended 0.8915 era.
    _dates = pd.to_datetime(d2[schema.DATE])
    _key = _dates.map(lambda d: tuple(config.regime_on("v2", d).get(k) for k in ("from", "until")))
    _keep = _key.value_counts().idxmax()
    regime = next(r for r in config.DECISION_RULES["v2"]["regimes"]
                  if (r.get("from"), r.get("until")) == _keep)
    regime_tau = regime["threshold"]
    d2 = d2[_key.eq(_keep)]
    print(f"regime kept: {regime.get('from', '(open)')} -> {regime.get('until', '(open)')} "
          f"tau={regime_tau} | rows {len(d2):,} of {len(_key):,} "
          f"({_key.nunique()} regime(s) present in the join)")

    gv = d2[d2[schema.DECISION].eq(0)].copy()          # garage-verified only
    gv["Y"] = gv[schema.OBSERVED].astype(int)
    near = gv[gv[S2] > regime_tau - 0.15]
    print(f"v2 garage-verified single-regime rows joined with v3 scores: {len(gv):,}; "
          f"near-boundary (s_v2 > {regime_tau - 0.15:.3f}): {len(near):,}")

    edges2 = np.arange(regime_tau - 0.15, regime_tau + 1e-9, 0.015)
    gap = []
    for a, b in zip(edges2[:-1], edges2[1:]):
        seg = near[(near[S2] >= a) & (near[S2] < b)]
        if (seg["Y"] == 0).sum() >= MIN_N_SIDE and (seg["Y"] == 1).sum() >= MIN_N_SIDE:
            gap.append({"s_v2_mid": (a + b) / 2,
                        "gap": seg.loc[seg.Y == 1, S3].mean() - seg.loc[seg.Y == 0, S3].mean(),
                        "n0": int((seg.Y == 0).sum()), "n1": int((seg.Y == 1).sum())})
    gap = pd.DataFrame(gap)
    if len(gap) >= 4:
        slope = np.polyfit(gap["s_v2_mid"], gap["gap"], 1)[0]
        print(gap)
        print(f"\ngap slope toward the cutoff: {slope:+.4f} per unit s_v2 "
              f"(negative = gap narrowing = consistent with inheritance; interpret only with §3.3.1 caveat)")
    else:
        print("Too few populated bins near the boundary — report as underpowered.")
except FileNotFoundError as e:
    print("v2→v3 analogue skipped — missing artefact:\n", e)

## 7 · What goes into the paper

- `[[TAB 4.0]]`: the `results` table of §4 (beta, CI, placebo pass/fail, n per row) plus the §6
  gap-slope with its caveat line.
- `[[FIG 4.0]]`: `figures/v1v2_01_error_inheritance.png`.
- If the n-gate failed: the §3 counts table **is** the result — "underpowered, n = …" — and no
  beta is quoted anywhere.
- Every quoted beta carries: the band definition from `config.DECISION_RULES['v1']`, the
  mobility-confound reminder (mobile band rows are systematically more repairable — the *level*
  of `Y=0` share is confounded; the *jump in v2's score at the cutoff within the mobile verified
  population* is the estimate), and the placebo verdicts.